# Full Pipeline Demo: Satellite DB build + 5 random UAV queries (Flight 01)

End-to-end walkthrough of the method on UAV-VisLoc flight 01:

1. Clone the repo, install dependencies, pull the trained Mask R-CNN checkpoint and the UAV-VisLoc dataset.
2. Build the **satellite KD-Tree database** from `satellite01.tif` (9774 x 26762) using the paper's 500 px patch / 100-px stride setup.
3. Pick **5 random UAV images** from flight 01's drone folder.
4. For each: preprocess, extract 24-D CFBVM-PF descriptors, KD-Tree query, plurality vote, render the **top-100 ranked candidate patches** on the satellite map (each labeled with its rank 1..100), plus the GT marker.

See [TUTORIAL.md](../TUTORIAL.md) for a deeper explanation of each step.


## 1. Clone the repo (Colab / Kaggle)

Skip if you are already running from a cloned checkout.

In [ ]:
# import os

# REPO_URL = 'https://github.com/kagtgi/LocalizationUAV.git'
# REPO_DIR = 'LocalizationUAV'

# if not os.path.exists(REPO_DIR) and not os.path.exists('localization'):
#     !git clone {REPO_URL} {REPO_DIR}
#     %cd {REPO_DIR}
# !pwd


## 2. Install dependencies

In [ ]:
# !pip install --quiet -r requirements.txt

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('/home/nguyenduytan/LocalizationUAV')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / 'UAV_VisLoc_dataset'
FLIGHT_ID = '09'
MODEL_PATH = REPO_ROOT / 'best_model.pth'

PATCH_SIZE = 500
STRIDE = 100
SCORE_THRESHOLD = 0.5
INFERENCE_BATCH_SIZE = 4
TOP_K = 5
REFERENCE_RADII_M = (20.0, 40.0, 60.0)
METERS_PER_PIXEL   = 0.3
TOP_N_TO_SHOW = 100
NUM_RANDOM_SAMPLES = 5
RANDOM_SEED = 42

OUTPUT_DIR = REPO_ROOT / 'outputs' / FLIGHT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_NPZ_PATH          = OUTPUT_DIR / f'satellite{FLIGHT_ID}_cfbvm_kdtree.npz'
DESCRIPTORS_CSV_PATH = OUTPUT_DIR / f'satellite{FLIGHT_ID}_cfbvm_descriptors.csv'

# ── Flight 09: 4 TIF tiles arranged in a 2×2 grid (row × col) ────────────
# Naming convention: satellite09_{row:02d}-{col:02d}.tif  (1-indexed)
#   row 1: top half    row 2: bottom half
#   col 1: left half   col 2: right half
SAT_TIFS = {
    (0, 0): DATA_ROOT / FLIGHT_ID / f'satellite{FLIGHT_ID}_01-01.tif',  # top-left
    (0, 1): DATA_ROOT / FLIGHT_ID / f'satellite{FLIGHT_ID}_01-02.tif',  # top-right
    (1, 0): DATA_ROOT / FLIGHT_ID / f'satellite{FLIGHT_ID}_02-01.tif',  # bottom-left
    (1, 1): DATA_ROOT / FLIGHT_ID / f'satellite{FLIGHT_ID}_02-02.tif',  # bottom-right
}
IS_MOSAIC = True   # set False for single-TIF flights

print('Repo root :', REPO_ROOT)
print('Data root :', DATA_ROOT)
print('Model     :', MODEL_PATH)
print('DB output :', DB_NPZ_PATH)
if IS_MOSAIC:
    for pos, p in SAT_TIFS.items():
        print(f'  tile {pos} : {p}  exists={p.exists()}')


In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

from localization import (
    load_model,
    process_uav,
    SatelliteDatabase,
    build_satellite_descriptors,
    query_uav,
)
from localization.database.builder import extract_patch_descriptors
from localization.io.bounds import (
    load_satellite_bounds,
    latlon_to_pixel,
    pixel_to_latlon,
    pixel_offset_to_meters,
)
from localization.io.dataset import VisLocFlight, load_flight_metadata, get_image_pose
from localization.matching.visualize import render_top_n_result, draw_gt_vs_topn_centroids

flight = VisLocFlight(flight_id=FLIGHT_ID, root=DATA_ROOT)
print('Metadata CSV  :', flight.metadata_csv)
print('Bounds CSV    :', flight.bounds_csv)
assert MODEL_PATH.exists(), f'Missing checkpoint: {MODEL_PATH}'

if IS_MOSAIC:
    # Đọc kích thước từng tile để tính tổng kích thước mosaic
    _tile_sizes = {}
    for pos, tif in SAT_TIFS.items():
        assert tif.exists(), f'Missing tile: {tif}'
        with Image.open(tif) as _im:
            _tile_sizes[pos] = _im.size   # (w, h)

    # Giả sử tất cả tile cùng kích thước (hợp lệ cho UAV-VisLoc)
    _tw, _th = _tile_sizes[(0, 0)]
    _n_tile_rows = max(r for r, c in SAT_TIFS) + 1
    _n_tile_cols = max(c for r, c in SAT_TIFS) + 1
    sat_w = _tw * _n_tile_cols
    sat_h = _th * _n_tile_rows

    print(f'Tile size     : {_tw} × {_th}')
    print(f'Tile grid     : {_n_tile_rows} rows × {_n_tile_cols} cols')
    print(f'Mosaic size   : {sat_w} × {sat_h} px  (virtual, không ghi disk)')

    # Helper: crop một vùng pixel từ mosaic (lazy, không load toàn bộ)
    def mosaic_crop(x0: int, y0: int, x1: int, y1: int) -> Image.Image:
        """Crop [x0,x1) × [y0,y1) từ virtual mosaic (tọa độ mosaic)."""
        out = Image.new('RGB', (x1 - x0, y1 - y0))
        for (tr, tc), tif in SAT_TIFS.items():
            tx0, ty0 = tc * _tw, tr * _th   # pixel offset của tile trong mosaic
            tx1, ty1 = tx0 + _tw, ty0 + _th
            # Tìm vùng giao giữa crop-rect và tile
            ix0, iy0 = max(x0, tx0), max(y0, ty0)
            ix1, iy1 = min(x1, tx1), min(y1, ty1)
            if ix0 >= ix1 or iy0 >= iy1:
                continue   # không giao
            with Image.open(tif) as _t:
                region = _t.crop((ix0 - tx0, iy0 - ty0,
                                  ix1 - tx0, iy1 - ty0)).convert('RGB')
            out.paste(region, (ix0 - x0, iy0 - y0))
        return out

    # Dummy path cho các hàm cần satellite_image_path (chỉ dùng trong visualize)
    # render_top_n_result nhận path → đọc file; ta sẽ truyền tile 0,0 và chú thích
    SAT_VIZ_TIF = str(SAT_TIFS[(0, 0)])   # dùng tạm cho bước visualize

else:
    assert flight.satellite_tif.exists(), f'Missing TIF: {flight.satellite_tif}'
    with Image.open(flight.satellite_tif) as _im:
        sat_w, sat_h = _im.size
    mosaic_crop = None
    SAT_VIZ_TIF = str(flight.satellite_tif)
    print(f'Satellite TIF : {flight.satellite_tif}  ({sat_w}×{sat_h})')

print(f'sat_w={sat_w}, sat_h={sat_h}')


## 4. Download the Mask R-CNN checkpoint from Google Drive

`best_model.pth` (~170 MB) is hosted at https://drive.google.com/file/d/1jlRfOXYU18DcOjWEwNBet1b7FHCIClcB/view

In [ ]:
import os

# Flight 09: bounds CSV có thể không có entry cho mosaic ghép
# → thử tìm bounds từ tile 01-01; nếu không có thì bounds=None
if IS_MOSAIC:
    bounds = load_satellite_bounds(
        satellite_filename=f'satellite{FLIGHT_ID}_01-01.tif',
        csv_path=str(flight.bounds_csv),
    )
    if bounds is None:
        # Thử key không có suffix tile
        bounds = load_satellite_bounds(
            satellite_filename=f'satellite{FLIGHT_ID}.tif',
            csv_path=str(flight.bounds_csv),
        )
else:
    bounds = load_satellite_bounds(
        satellite_filename=os.path.basename(str(flight.satellite_tif)),
        csv_path=str(flight.bounds_csv),
    )

print('Bounds:', bounds)
print(f'Satellite size: {sat_w} x {sat_h}')

metadata_df = load_flight_metadata(flight.metadata_csv)
print('Metadata rows :', len(metadata_df))

## 6. Load the Mask R-CNN backbone

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model(
    model_path=str(MODEL_PATH),
    device=device,
    num_classes=2,
    pretrained=False,
).to(device).eval()
print('Device:', device)


## 7. Build (or load) the satellite KD-Tree database — **chunked**

Satellite lớn (ví dụ flight 03) có thể có **> 25 000 patches** — tích lũy toàn bộ vào RAM
trước khi ghi dễ gây OOM. Giải pháp: **chia theo vertical strip** (nhóm cột patch).

```
satellite.tif  (W × H)
┌──────┬──────┬──────┐
│strip │strip │strip │  ← mỗi strip = CHUNK_COLS cột patch × toàn bộ chiều cao
│  0   │  1   │  2   │    xử lý xong → flush CSV → del → gc → strip tiếp theo
└──────┴──────┴──────┘
```

Mỗi strip được crop từ TIF gốc bằng PIL (lazy-load, chỉ decode vùng cần). Patch `(col, row)`
được đánh ID và tính centroid theo **tọa độ pixel ảnh gốc** — không bị lệch.

Sau khi tất cả strips xong, đọc lại CSV → build KD-Tree một lần → lưu `.npz`.

| Tham số | Mặc định | Ghi chú |
|---------|----------|---------|
| `CHUNK_COLS` | `20` | Số cột patch mỗi strip (giảm → ít RAM hơn) |


In [ ]:
import os, gc, math
import pandas as pd
from PIL import Image as _PILImage
from tqdm.auto import tqdm

_PILImage.MAX_IMAGE_PIXELS = None

CHUNK_COLS = 20

_FEAT_COLS = tuple(
    f's{i}{j}'
    for i in range(1, 4)
    for j in range(1, 9)
)

if DB_NPZ_PATH.exists():
    print('Loading existing database:', DB_NPZ_PATH)
    db = SatelliteDatabase.load(str(DB_NPZ_PATH))
    print(db)
else:
    _n_cols   = math.ceil(max(sat_w - PATCH_SIZE, 0) / STRIDE) + 1
    _n_rows   = math.ceil(max(sat_h - PATCH_SIZE, 0) / STRIDE) + 1
    _n_chunks = math.ceil(_n_cols / CHUNK_COLS)
    _total_patches = _n_cols * _n_rows
    print(f'Mosaic         : {sat_w} × {sat_h} px')
    print(f'Patch grid     : {_n_cols} cols × {_n_rows} rows = {_total_patches:,} patches → {_n_chunks} strips')

    DESCRIPTORS_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    if DESCRIPTORS_CSV_PATH.exists():
        DESCRIPTORS_CSV_PATH.unlink()

    _total_rows = 0
    _pbar = tqdm(total=_total_patches, desc='Building DB', unit='patch', dynamic_ncols=True)

    for _ci in range(_n_chunks):
        _c0      = _ci * CHUNK_COLS
        _c1      = min(_c0 + CHUNK_COLS, _n_cols)
        _px0     = _c0 * STRIDE
        _px1     = min(_c1 * STRIDE + PATCH_SIZE, sat_w)
        _strip_w = _px1 - _px0

        _pbar.set_postfix(strip=f'{_ci+1}/{_n_chunks}', rows=f'{_total_rows:,}', refresh=False)

        # ── Crop strip: mosaic hoặc single TIF ──────────────────────────────
        if IS_MOSAIC:
            _strip = mosaic_crop(_px0, 0, _px1, sat_h)
        else:
            with _PILImage.open(flight.satellite_tif) as _sat_img:
                _strip = _sat_img.crop((_px0, 0, _px1, sat_h)).convert('RGB')

        _rows_this_strip = []

        for _row in range(_n_rows):
            _py = _row * STRIDE
            for _col in range(_c0, _c1):
                _px_in_strip = (_col - _c0) * STRIDE
                if _px_in_strip + PATCH_SIZE > _strip_w:
                    _pbar.update(1)
                    continue

                _patch = _strip.crop((
                    _px_in_strip,              _py,
                    _px_in_strip + PATCH_SIZE, _py + PATCH_SIZE,
                ))
                _pid = f'r{_row:05d}_c{_col:05d}'
                _cx  = float(_px0 + _px_in_strip + PATCH_SIZE / 2)
                _cy  = float(_py + PATCH_SIZE / 2)

                _descs, _anchors = extract_patch_descriptors(
                    patch_image       = _patch,
                    model             = model,
                    device            = device,
                    score_threshold   = SCORE_THRESHOLD,
                    reference_radii_m = REFERENCE_RADII_M,
                    meters_per_pixel  = METERS_PER_PIXEL,
                )
                _pbar.update(1)

                if _descs.shape[0] == 0:
                    continue

                _ax = _anchors[:, 0] + (_px0 + _px_in_strip)
                _ay = _anchors[:, 1] + _py

                for _k in range(len(_descs)):
                    _r = {'patch_id': _pid, 'cx': _cx, 'cy': _cy,
                          'ax': float(_ax[_k]), 'ay': float(_ay[_k])}
                    for _fname, _fval in zip(_FEAT_COLS, _descs[_k]):
                        _r[_fname] = float(_fval)
                    _rows_this_strip.append(_r)

        del _strip
        gc.collect()
        if device.type == 'cuda':
            import torch as _torch
            _torch.cuda.empty_cache()

        if _rows_this_strip:
            _strip_df     = pd.DataFrame(_rows_this_strip)
            _write_header = not DESCRIPTORS_CSV_PATH.exists()
            _strip_df.to_csv(str(DESCRIPTORS_CSV_PATH), mode='a',
                             header=_write_header, index=False)
            _total_rows += len(_strip_df)
            _pbar.set_postfix(strip=f'{_ci+1}/{_n_chunks}', rows=f'{_total_rows:,}', refresh=True)
            del _strip_df
        gc.collect()

    _pbar.close()
    print(f'\nTổng {_total_rows:,} descriptors. Build KD-Tree...')

    descriptors_df = pd.read_csv(str(DESCRIPTORS_CSV_PATH))
    print(f'Unique patches : {descriptors_df["patch_id"].nunique():,}')

    db = SatelliteDatabase.from_dataframe(
        descriptors_df,
        parent_tif = f'satellite{FLIGHT_ID}_mosaic',
        leaf_size  = 40,
    )
    db.save(str(DB_NPZ_PATH))
    print('KD-Tree saved :', DB_NPZ_PATH)
    del descriptors_df
    gc.collect()
    print(db)


## 8. Pick 5 random UAV images from flight 01

In [ ]:
import random

drone_images = sorted(p for p in flight.drone_dir.iterdir()
                      if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print(f'{len(drone_images)} drone images available in flight {FLIGHT_ID}')
assert len(drone_images) > 0, 'No drone images found.'

random.seed(RANDOM_SEED)
samples = random.sample(drone_images, k=min(NUM_RANDOM_SAMPLES, len(drone_images)))
for s in samples:
    print('  picked:', s.name)


## 9. Load satellite bounds + flight metadata (shared by all queries)

In [ ]:
import os

bounds = load_satellite_bounds(
    satellite_filename=os.path.basename(str(flight.satellite_tif)),
    csv_path=str(flight.bounds_csv),
)
print('Bounds:', bounds)

with Image.open(flight.satellite_tif) as sat:
    sat_w, sat_h = sat.size
print(f'Satellite size: {sat_w} x {sat_h}')

metadata_df = load_flight_metadata(flight.metadata_csv)
print('Metadata rows :', len(metadata_df))


## 10. Per-sample query loop

For each of the 5 random samples:

1. Preprocess the UAV image to 500 x 500 (yaw + altitude + crop).
2. Run Mask R-CNN → contour → CFBVM-PF → 24-D descriptors (same `segment_batch` used for the satellite patches).
3. KD-Tree query with K=5, plurality vote, build the top-`TOP_N_TO_SHOW` ranked list.
4. Convert rank-1 pixel → lat/lon and compute the error vs GT.
5. Render the 3-panel figure with all 100 ranked candidates labeled, plus GT.

Outputs land in `outputs/01/match_top100_<image>.png`.

In [ ]:
import matplotlib.pyplot as plt

results_summary = []

for sample_path in samples:
    image_name = sample_path.name
    print('=' * 70)
    print('Processing:', image_name)

    # Step 1: preprocess to 500x500
    _, img_uav_500, meta = process_uav(
        img_path=str(sample_path),
        csv_path=str(flight.metadata_csv),
    )
    print(f'  Pose: yaw={meta["Yaw (Phi)"]:.1f}, height={meta["height"]:.1f}')

    # Steps 2-4: descriptors
    uav_descriptors, _ = extract_patch_descriptors(
        patch_image=img_uav_500,
        model=model,
        device=device,
        score_threshold=0.5,
        reference_radii_m=REFERENCE_RADII_M,  # CFBVM-PF annuli radii
        meters_per_pixel=METERS_PER_PIXEL,     # satellite tile scale
    )
    if uav_descriptors.shape[0] == 0:
        print('  No buildings segmented; skipping this sample.')
        results_summary.append({'image': image_name, 'descriptors': 0, 'error_m': None,
                               'rank1_patch': None, 'rank1_votes': None, 'margin': None})
        continue
    print(f'  UAV descriptors: {uav_descriptors.shape[0]}')

    # Step 5: KD-tree query + plurality vote + top-N list
    result = query_uav(uav_descriptors, db, k=TOP_K, top_n=TOP_N_TO_SHOW)
    if result is None:
        print('  Query produced no winner; skipping.')
        continue
    print(f'  Rank-1 patch  : {result.patch_id}  (votes={result.vote_count}, margin={result.margin})')

    # GT lookup + error in meters
    gt_pixel = None
    error_distance_m = None
    row = get_image_pose(metadata_df, image_name)
    if row is not None and bounds is not None:
        gt_lat, gt_lon = float(row['lat']), float(row['lon'])
        gt_pixel = latlon_to_pixel(gt_lat, gt_lon, bounds, sat_w, sat_h)
        offset = pixel_offset_to_meters(
            result.pixel_xy[0] - gt_pixel[0],
            result.pixel_xy[1] - gt_pixel[1],
            bounds, sat_w, sat_h,
        )
        error_distance_m = float(offset['distance_m'])
        print(f'  GT pixel      : {gt_pixel}')
        print(f'  Error vs GT   : {error_distance_m:.1f} m '
              f'(dx={offset["dx_m"]:.1f}, dy={offset["dy_m"]:.1f})')

    # Visualize top-N (3-panel)
    out_path = OUTPUT_DIR / f'match_top{TOP_N_TO_SHOW}_{sample_path.stem}.png'
    title = f'Top {TOP_N_TO_SHOW} candidates: {image_name}  ->  rank-1 = {result.patch_id}'
    if error_distance_m is not None:
        title += f'   (error: {error_distance_m:.1f} m)'

    fig = render_top_n_result(
        uav_image_path=str(sample_path),
        satellite_image_path=str(flight.satellite_tif),
        top_n=result.top_n,
        gt_pixel_xy=gt_pixel,
        error_distance_m=error_distance_m,
        zoom_radius_px=1500,
        highlight_top_k=5,
        title=title,
        output_path=str(out_path),
    )
    plt.show()
    plt.close(fig)
    print('  Figure saved to:', out_path)

    # Paper Fig. 5(a)-style "GT vs top-N matches centroids" map.
    fig5a_title = (
        f'Satellite map ({FLIGHT_ID}, {meta["height"]:.0f} m, '
        f'phi={meta["Yaw (Phi)"]:.0f} deg)'
    )
    gt_vs_top_path = OUTPUT_DIR / f'gt_vs_top{TOP_N_TO_SHOW}_{sample_path.stem}.png'
    fig5a = draw_gt_vs_topn_centroids(
        satellite_image_path=str(flight.satellite_tif),
        top_n=result.top_n,
        gt_pixel_xy=gt_pixel,
        title=fig5a_title,
        output_path=str(gt_vs_top_path),
    )
    plt.show()
    plt.close(fig5a)
    print('  GT-vs-top-N map saved to:', gt_vs_top_path)

    results_summary.append({
        'image': image_name,
        'descriptors': int(uav_descriptors.shape[0]),
        'rank1_patch': result.patch_id,
        'rank1_votes': int(result.vote_count),
        'margin': int(result.margin),
        'error_m': error_distance_m,
    })

print('=' * 70)
print('All samples processed.')


## 11. Summary table

In [ ]:
summary_df = pd.DataFrame(results_summary)
summary_df


In [ ]:
ok = summary_df['error_m'].notna()
if ok.any():
    print('Error vs GT (m) over', int(ok.sum()), 'samples:')
    print('  mean   :', float(summary_df.loc[ok, 'error_m'].mean()))
    print('  median :', float(summary_df.loc[ok, 'error_m'].median()))
    print('  min    :', float(summary_df.loc[ok, 'error_m'].min()))
    print('  max    :', float(summary_df.loc[ok, 'error_m'].max()))


Done. Per-image figures live in `outputs/01/match_top100_*.png` for inspection.